In [14]:
from ase.build import fcc100, bcc100, fcc111, bcc110
from ase.io import write
import numpy as np
from ase.visualize import view
import os
os.environ["OMP_NUM_THREADS"] = "4"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["OMP_PLACES"] = "cores"
os.environ["OMP_PROC_BIND"] = "close"

# Lattice constants
a_cu = 3.615
a_ta = 3.300

# slab sizes
nx, ny = 1, 1
cu_layers = 4
ta_layers = 4

# build slabs
cu = fcc100('Cu', size=(nx, ny, cu_layers), a=a_cu, vacuum=0) #, orthogonal=True, periodic=True)
ta = bcc100('Ta', size=(nx, ny, ta_layers), a=a_ta, vacuum=0.0)
write("cu.vasp", cu)
print(cu)



Atoms(symbols='Cu4', pbc=[True, True, False], cell=[2.5561910139893698, 2.5561910139893698, 5.4225], tags=...)


In [15]:
# match Ta's in-plane cell to Cu
cu_cell = cu.get_cell()
ta_cell = ta.get_cell()
print(cu_cell)
print(ta_cell)
new_ta_cell = ta_cell.copy()
new_ta_cell[0] = cu_cell[0]
new_ta_cell[1] = cu_cell[1]
ta.set_cell(new_ta_cell, scale_atoms=True)
ta_cell = ta.get_cell()

print(ta_cell)
view(cu, viewer="ngl")


Cell([2.5561910139893698, 2.5561910139893698, 5.4225])
Cell([3.3, 3.3, 4.949999999999999])
Cell([2.5561910139893698, 2.5561910139893698, 4.949999999999999])


In [4]:
# shift slabs
vacuum = 8.0

cu_zmax = cu.get_positions()[:,2].max()
ta_positions = ta.get_positions()
ta_positions[:,2] -= ta_positions[:,2].min()
ta_positions[:,2] += cu_zmax + vacuum  # small interface gap
ta.set_positions(ta_positions)
print(ta.get_positions())
view(ta, viewer="ngl")




[[ 0.          2.21372636 14.26136367]
 [ 0.          0.         16.59481605]
 [ 0.          2.21372636 18.92826843]
 [ 0.          0.         21.2617208 ]]


In [5]:
# combine
interface = cu + ta
print(interface.get_cell())
view(interface, viewer="ngl")

Cell([[2.5561910139893698, 0.0, 0.0], [1.2780955069946849, 2.213726355040297, 0.0], [0.0, 0.0, 6.261363669361492]])


In [6]:
# add vacuum in z

zmin = interface.positions[:,2].min()
zmax = interface.positions[:,2].max()
Lz = (zmax - zmin) + vacuum

cell = np.array([
    cu_cell[0],
    cu_cell[1],
    [0, 0, Lz]
])
interface.set_cell(cell)
#interface.center(axis=2)
print(interface.get_cell())
view(interface, viewer="ngl")



Cell([[2.5561910139893698, 0.0, 0.0], [1.2780955069946849, 2.213726355040297, 0.0], [0.0, 0.0, 29.26172080310831]])


In [7]:
# write
write("cu_ta_fcc111_bcc110.cif", interface)
write("cu_ta_fcc111_bcc110.lmp", interface, format='lammps-data', atom_style='atomic')
write("cu_ta_fcc111_bcc110.vasp", interface)

print("Done. Atoms:", len(interface))

Done. Atoms: 8


In [8]:
from ase.calculators.espresso import Espresso, EspressoProfile
profile = EspressoProfile(
    command='mpirun -np 4 pw.x', pseudo_dir='/home/jovyan/NSCI0032-25-26-tutor/cuta_interface/pseudo')
pseudopotentials = {
    'Cu': 'Cu.pbe-dn-kjpaw_psl.1.0.0.UPF',
    'Ta': 'Ta_pbe_v1.uspp.F.UPF',
} # a dict!

input_data = {
    'control': {
        'calculation': 'scf',
        'verbosity': 'high',
    },
    'system': {
        'ecutwfc': 50,    # Plane-wave cutoff energy in Ry
        'ecutrho': 400,   # Charge density cutoff in Ry
        'occupations': 'smearing',
        'smearing': 'mp',
        'degauss': 0.02,  # Width of smearing in Ry
    },
    'electrons': {
        'conv_thr': 1e-6,  # Convergence threshold
    },
}
kpts = (4, 4, 1)  # k-point grid for the slab

In [9]:
calc = Espresso(profile=profile, pseudopotentials=pseudopotentials,
                tstress=False, tprnfor=False, kpts=kpts, input_data=input_data)

In [ ]:
interface.calc = calc

# Calculate bulk energy per atom
interface_energy = interface.get_potential_energy()
print(interface_energy)